# Cascaded calibration with ruflo agents

**Thesis:** `lub` quantifies *when* a model is trustworthy. [ruflo](https://github.com/ruvnet/ruflo) orchestrates *which* model runs. Combining them produces an **uncertainty-governed agent swarm** — the missing piece for banks deploying LLMs at scale without abandoning model risk management.

This notebook demonstrates the synthesis end-to-end using only `lub.orchestration` modules that already ship in v0.1. No external `ruflo` dependency required — we use a thin `RufloAgentStub` so the notebook is hermetic and CI-runnable. The same shape works against the real ruflo runtime in v0.3 via `lub.agents.adapters.ruflo` (RFC-001).

**You'll see three things compose:**
1. `TieredRouter` — cascaded model dispatch (cheap → expensive) gated by calibrated confidence per tier
2. `UQSwarm` — DAA-style consensus across multiple uncertainty estimators on the same prompt
3. `RufloAgentStub` — wraps the calibrated pipeline into an agent-style interface (`act(observation) -> Action | Refusal`)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

from lub.orchestration import TieredRouter, Tier, UQSwarm
from lub.pipeline import UncertaintyPipeline
from lub.uncertainty.token_logprob import TokenLogprobEstimator
from lub.uncertainty.self_consistency import SelfConsistencyEstimator
from lub.uncertainty.p_true import PTrueEstimator
from lub.wrappers.dummy import DummyBackend

print("lub imports OK")

## 1. Build three tiers (cheap → strong)

Each tier has its own pipeline, threshold, and cost. The router escalates only when confidence falls below the tier's threshold. Real deployments would use Llama-3-8B → Claude Haiku → Claude Sonnet; here we use three deterministic `DummyBackend` instances labeled with distinct `model_id`s so the audit trail records which tier ran.

In [ ]:
def make_tier(name: str, model_id: str, threshold: float, cost: float) -> Tier:
    backend = DummyBackend(model_id=model_id)
    pipeline = UncertaintyPipeline(
        backend=backend,
        estimator=TokenLogprobEstimator(),
    )
    return Tier(name=name, pipeline=pipeline, threshold=threshold, cost=cost)

tiers = [
    make_tier("llama3-8b", model_id="dummy-llama3-8b", threshold=0.85, cost=0.0001),
    make_tier("haiku",     model_id="dummy-haiku",     threshold=0.75, cost=0.0025),
    make_tier("sonnet",    model_id="dummy-sonnet",    threshold=0.0,  cost=0.015),  # final tier, never escalates
]

router = TieredRouter(tiers=tiers)
print(f"Router built with {len(tiers)} tiers")

## 2. Build the UQ swarm

Three estimators score the *same* prompt and the swarm fuses them. Disagreement between methods is itself a signal — high `method_disagreement` means the methods can't agree, which is exactly the case where a human reviewer should look.

In [ ]:
swarm_backend = DummyBackend(model_id="dummy-swarm")

swarm = UQSwarm(
    backend=swarm_backend,
    estimators={
        "token_logprob":    TokenLogprobEstimator(),
        "self_consistency": SelfConsistencyEstimator(n_samples=5),
        "p_true":           PTrueEstimator(),
    },
    # Equal weights by default. In production these come from the ledger
    # by replaying historical reliability per method.
)
print("Swarm assembled with 3 estimators")

## 3. The `RufloAgentStub`

This stub stands in for `lub.agents.adapters.ruflo.RufloAgent` (RFC-001, target v0.3). The contract is the same: an `act(observation)` method that returns an `Action` or a `Refusal` based on calibrated confidence. The real adapter will plug into ruflo's swarm coordinator; this stub captures the shape so the demo is end-to-end without the upstream dep.

In [ ]:
@dataclass(frozen=True)
class Action:
    answer: str
    confidence: float
    tier_used: str
    method_disagreement: float
    cost: float

@dataclass(frozen=True)
class Refusal:
    reason: str
    confidence: float
    method_disagreement: float

class RufloAgentStub:
    """Wraps a TieredRouter + UQSwarm into an agent-shaped interface.

    The real adapter will live at ``lub.agents.adapters.ruflo.RufloAgent``
    once the v0.3 RFC lands. Contract is the same: ``act(observation)``
    returns either an :class:`Action` or a :class:`Refusal`.
    """

    def __init__(self, router: TieredRouter, swarm: UQSwarm,
                 refusal_threshold: float = 0.6,
                 disagreement_threshold: float = 0.25):
        self.router = router
        self.swarm = swarm
        self.refusal_threshold = refusal_threshold
        self.disagreement_threshold = disagreement_threshold

    def act(self, observation: str) -> Action | Refusal:
        # Route through tiers (cheapest first)
        routed = self.router.answer(observation)

        # Independent second-opinion: ask the swarm to score the same prompt
        swarm_out = self.swarm.answer(observation)
        disagreement = swarm_out.fused.raw_scores["method_disagreement"]
        fused_conf = swarm_out.fused.confidence

        # Two refusal gates
        if fused_conf < self.refusal_threshold:
            return Refusal(reason="swarm-fused confidence below threshold",
                          confidence=fused_conf, method_disagreement=disagreement)
        if disagreement > self.disagreement_threshold:
            return Refusal(reason="UQ methods disagree — human review recommended",
                          confidence=fused_conf, method_disagreement=disagreement)

        return Action(answer=routed.final.answer, confidence=fused_conf,
                     tier_used=routed.tier_used, method_disagreement=disagreement,
                     cost=routed.total_cost)

agent = RufloAgentStub(router=router, swarm=swarm)
print("Agent wired up")

## 4. Run a regulated-domain question

These prompts come from the BR-Regulatory benchmark (BCB Resolution 4.658 + Basel III). The interesting thing isn't whether the dummy backend gets the answer right — it's the *decision trace*: which tier fired, what the swarm fusion produced, and whether the agent acted or refused.

In [ ]:
prompts = [
    "What is the Basel III minimum CET1 ratio?",
    "What does BCB Resolution 4.658 require for cyber incident notification?",
    "Should the bank approve a loan to Customer X?",  # out-of-domain on purpose
]

for prompt in prompts:
    decision = agent.act(prompt)
    print(f"\nQ: {prompt}")
    if isinstance(decision, Action):
        print(f"  ACT    | tier={decision.tier_used} conf={decision.confidence:.2f} "
              f"disagreement={decision.method_disagreement:.2f} cost=${decision.cost:.4f}")
        print(f"         | {decision.answer[:80]}")
    else:
        print(f"  REFUSE | {decision.reason} (conf={decision.confidence:.2f}, "
              f"disagreement={decision.method_disagreement:.2f})")

## 5. Why this matters for model-risk review

Every decision the agent makes carries a four-field receipt:

| Field | What it tells the auditor |
|---|---|
| `tier_used` | Which model ultimately answered (cost & capability tier) |
| `confidence` | Swarm-fused calibrated confidence in [0, 1] |
| `method_disagreement` | Second-order signal: how much the UQ methods disagree |
| `cost` | Total monetary cost across all tiers consulted |

Pair this with `lub.ledger.Ledger.log_policy(decision)` (already shipping in v0.1) and you get a per-decision audit trail that satisfies SR 11-7 and BCB Resolução 4.893 model-risk documentation requirements without manual log scraping.

## 6. What's stubbed vs. what ships

**Already in v0.1** (used directly above):
- `lub.orchestration.TieredRouter` — cascaded routing
- `lub.orchestration.UQSwarm` — multi-method fusion
- `lub.orchestration.HookedPipeline` — pre/post-answer hooks
- `lub.ledger.Ledger` — SQLite uncertainty ledger
- `lub.evidence.KNNStore` — retrieval-augmented attribution
- `lub.governance.drift` — drift enforcer
- `lub.mcp` — MCP tool surface

**Stubbed here, lands in v0.3** (RFC-001):
- `lub.agents.CalibratedAgent` — base class
- `lub.agents.RefusalPolicy` — composable refusal rules (replaces inline thresholds)
- `lub.agents.adapters.ruflo.RufloAgent` — plugs into ruflo's actual swarm coordinator
- `lub.agents.adapters.langgraph` / `crewai` / `autogen` — sibling adapters
- `lub.agents.ReportingAgent` — audit-trail-emitting base

See `planning/RFC_001_calibrated_agents_2026-04-23.md` for the full v0.3 design and `planning/12_Implementation_Prompts.md` for the ten execution prompts that land it.